# 00 · Recon — what data actually exists over Mussafah?

**Run this first. It takes ~10 minutes and answers four questions:**

1. How many clear **Landsat 8/9** scenes cover our AOI in each epoch? (drives the heat analysis)
2. How many clear **Sentinel-2** scenes? (drives albedo and built-up)
3. Does **ESA WorldCover** cover the AOI? (land cover baseline)
4. Does any **Planet Tanager** hyperspectral `urban` scene overlap the UAE? (decides whether we can claim the hyperspectral bonus)

Nothing here downloads imagery — it only queries catalogues and reads metadata.
Record the printed numbers; they go straight into the README's *Data used* section.


In [ ]:
%pip install -q pystac-client==0.8.6 planetary-computer==1.0.0 requests shapely geopandas

In [ ]:
# ---------------------------------------------------------------------------
# Bootstrap. Works in three situations, so you never have to think about paths:
#   1. Colab, notebook uploaded on its own  -> clones the repo and moves into it
#   2. Colab, repo already cloned           -> just moves into notebooks/
#   3. Local jupyter inside the repo        -> does nothing
# Set REPO_URL to your own repo once you have created it.
# ---------------------------------------------------------------------------
import os, sys, subprocess

REPO_URL  = "https://github.com/RISKYXBT/coolroof-mussafah.git"
REPO_DIR  = "coolroof-mussafah"

def _in_repo():
    return os.path.isdir("../results") and os.path.isdir("../data")

if not _in_repo():
    if os.path.isdir(f"/content/{REPO_DIR}/notebooks"):
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    elif "google.colab" in sys.modules:
        os.chdir("/content")
        if not os.path.isdir(REPO_DIR):
            print(f"Cloning {REPO_URL} ...")
            r = subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR],
                               capture_output=True, text=True)
            if r.returncode != 0:
                print("Clone failed:", r.stderr.strip())
                print("Falling back to a local working folder.")
                os.makedirs(f"{REPO_DIR}/notebooks", exist_ok=True)
                os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
                os.makedirs(f"{REPO_DIR}/data/sample_input", exist_ok=True)
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    else:
        # Local, but not inside the repo: create the folders beside us.
        os.makedirs("../results", exist_ok=True)
        os.makedirs("../data/sample_input", exist_ok=True)

print("working directory :", os.getcwd())
print("results folder    :", os.path.abspath("../results"))


In [ ]:
import json, requests
import pystac_client, planetary_computer
from shapely.geometry import box, shape

# ----------------------------------------------------------------------------
# AREA OF INTEREST
# Mussafah industrial area, Abu Dhabi. Large flat industrial/commercial roofs,
# strong thermal signal, and a decade of visible expansion into bare desert.
# bbox order is [min_lon, min_lat, max_lon, max_lat]
# ----------------------------------------------------------------------------
AOI_NAME = "Mussafah, Abu Dhabi"
AOI_BBOX = [54.43, 24.29, 54.58, 24.41]

# Secondary candidate if Mussafah disappoints (Khalifa Industrial Zone)
KIZAD_BBOX = [54.58, 24.70, 54.80, 24.88]

# Summer windows. UAE peak heat is June-August; we widen to May-September and
# span three years per epoch so enough cloud-free scenes accumulate.
EPOCH_A = "2014-05-01/2016-09-30"   # "before"
EPOCH_B = "2024-05-01/2026-09-30"   # "after"

aoi_geom = box(*AOI_BBOX)
print(f"AOI: {AOI_NAME}")
print(f"bbox: {AOI_BBOX}")
print(f"approx size: {(AOI_BBOX[2]-AOI_BBOX[0])*101:.1f} km x {(AOI_BBOX[3]-AOI_BBOX[1])*111:.1f} km")

## 1 · Landsat Collection 2 Level-2 — surface temperature

We use **Level-2**, not Level-1. Level-2 ships a USGS-produced, atmospherically
corrected **Surface Temperature** band (`lwir11`), so we do not hand-roll a
mono-window algorithm from top-of-atmosphere radiance. Fewer assumptions, fewer
places to be wrong, and far easier to defend to a judge.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def count_landsat(bbox, window, max_cloud=15):
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime=window,
        query={
            "eo:cloud_cover": {"lt": max_cloud},
            "platform": {"in": ["landsat-8", "landsat-9"]},
        },
    )
    items = list(search.items())
    return items

for label, window in [("EPOCH A (before)", EPOCH_A), ("EPOCH B (after)", EPOCH_B)]:
    items = count_landsat(AOI_BBOX, window)
    print(f"\n{label}  {window}")
    print(f"  clear scenes (<15% cloud): {len(items)}")
    if items:
        dates = sorted(i.properties["datetime"][:10] for i in items)
        print(f"  first: {dates[0]}   last: {dates[-1]}")
        print(f"  assets on a sample item: {sorted(items[0].assets)[:12]} ...")
        print(f"  has 'lwir11' (surface temperature): {'lwir11' in items[0].assets}")
        print(f"  has 'emis' (emissivity): {'emis' in items[0].assets}")

**What good looks like:** 20+ scenes per epoch. Under 8 and the median composite
gets noisy — widen the window or drop the cloud threshold to 25%.

## 2 · Sentinel-2 L2A — albedo and built-up indices

Sentinel-2 starts in 2015 and gives 10–20 m bands, so it carries the albedo layer
and the current-state built-up mapping. Landsat carries the long baseline.

In [ ]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=AOI_BBOX,
    datetime=EPOCH_B,
    query={"eo:cloud_cover": {"lt": 10}},
)
s2_items = list(search.items())
print(f"Sentinel-2 L2A clear scenes in EPOCH B: {len(s2_items)}")
if s2_items:
    it = s2_items[0]
    needed = ["B02", "B03", "B04", "B08", "B11", "B12", "SCL"]
    have = [b for b in needed if b in it.assets]
    print(f"  required bands present: {have}")
    print(f"  missing: {[b for b in needed if b not in it.assets]}")
    print(f"  sample date: {it.properties['datetime'][:10]}")

## 3 · ESA WorldCover — independent land-cover reference

WorldCover (10 m, 2020 and 2021) gives us a published, independently produced
land-cover map. We use it as a **reference to validate our own built-up
classification against** — which is exactly what the judging criterion *validity*
is asking for, and what the onboarding guide means by "state the reference data".

In [ ]:
for coll in ["esa-worldcover"]:
    try:
        s = catalog.search(collections=[coll], bbox=AOI_BBOX)
        items = list(s.items())
        print(f"{coll}: {len(items)} items")
        for i in items:
            print(f"   {i.id}  |  {i.properties.get('start_datetime', '')[:10]}  |  assets: {sorted(i.assets)}")
    except Exception as e:
        print(f"{coll}: FAILED -> {e}")

## 4 · Planet Tanager hyperspectral — is the bonus available?

This is the decisive one. Tanager is 426 bands at 30 m, and the hackathon awards
**bonus points for meaningful hyperspectral use**. The organizers' notebook 02
hard-codes four `urban` scene IDs. We check whether any of them — or any other
scene in the collection — overlaps the UAE.

If none do, we do **not** fake it. We state in the README that no Tanager urban
scene covered the AOI during the PoC window, and we use EnMAP or go without.
Honest negative results score under *validity*; invented ones get caught.

In [ ]:
TANAGER_BASE = "https://www.planet.com/data/stac/tanager-core-imagery"

# Scene IDs the organizers use in notebook 02 (collection: "urban")
KNOWN_URBAN_IDS = [
    "20250407_035509_25_4001",
    "20250515_080954_16_4001",
    "20250516_164837_16_4001",
    "20250412_111800_00_4001",
]

uae_box = box(51.0, 22.5, 56.5, 26.5)   # whole UAE, generous

def check_scene(collection, iid):
    url = f"{TANAGER_BASE}/{collection}/{iid}/{iid}.json"
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        item = r.json()
    except Exception as e:
        return {"id": iid, "error": str(e)}
    b = item["bbox"]
    geom = box(b[0], b[1], b[2], b[3])
    return {
        "id": iid,
        "date": item["properties"].get("datetime", "")[:10],
        "bbox": [round(v, 3) for v in b],
        "centroid": (round(geom.centroid.x, 3), round(geom.centroid.y, 3)),
        "intersects_uae": geom.intersects(uae_box),
        "intersects_aoi": geom.intersects(aoi_geom),
        "assets": sorted(item.get("assets", {}).keys()),
    }

print("Checking the four known Tanager 'urban' scenes:\n")
hits = []
for iid in KNOWN_URBAN_IDS:
    r = check_scene("urban", iid)
    if "error" in r:
        print(f"  {iid}: ERROR {r['error']}")
        continue
    flag = "  <-- UAE!" if r["intersects_uae"] else ""
    print(f"  {r['id']}  {r['date']}  centroid={r['centroid']}{flag}")
    if r["intersects_uae"]:
        hits.append(r)

print(f"\nScenes intersecting the UAE: {len(hits)}")
for h in hits:
    print(f"   {h['id']}  bbox={h['bbox']}  overlaps our AOI: {h['intersects_aoi']}")
    print(f"      assets: {h['assets']}")

### 4b · Browse the whole Tanager catalogue

The four IDs above are only examples. This walks the open collections looking for
anything over the Gulf. It is slower (a minute or two) but worth it — one UAE scene
changes our hyperspectral story completely.

In [ ]:
def try_catalog():
    """Attempt to list all items in the Tanager open collections."""
    out = {}
    for coll in ["urban", "coastal-water-bodies", "agriculture", "geology",
                 "methane", "co2", "fires", "forests"]:
        url = f"{TANAGER_BASE}/{coll}/catalog.json"
        try:
            r = requests.get(url, timeout=60)
            if r.status_code != 200:
                out[coll] = f"HTTP {r.status_code}"
                continue
            cat = r.json()
            links = [l for l in cat.get("links", []) if l.get("rel") == "item"]
            out[coll] = f"{len(links)} items"
        except Exception as e:
            out[coll] = f"error: {type(e).__name__}"
    return out

print("Tanager open collections:")
for k, v in try_catalog().items():
    print(f"  {k:24s} {v}")

print("\nIf 'urban' lists items above, the next cell scans them for UAE coverage.")

In [ ]:
# Scan every item in the urban collection for UAE overlap.
# Safe to skip if the catalog.json above did not resolve.
url = f"{TANAGER_BASE}/urban/catalog.json"
uae_scenes = []
try:
    cat = requests.get(url, timeout=60).json()
    item_links = [l for l in cat.get("links", []) if l.get("rel") == "item"]
    print(f"Scanning {len(item_links)} urban scenes...")
    for n, l in enumerate(item_links):
        href = l["href"]
        if not href.startswith("http"):
            href = f"{TANAGER_BASE}/urban/{href.lstrip('./')}"
        try:
            it = requests.get(href, timeout=30).json()
            b = it["bbox"]
            if box(b[0], b[1], b[2], b[3]).intersects(uae_box):
                uae_scenes.append((it["id"], it["properties"].get("datetime", "")[:10], b))
                print(f"  UAE HIT: {it['id']}  {it['properties'].get('datetime','')[:10]}  {b}")
        except Exception:
            pass
        if n % 25 == 0 and n:
            print(f"   ...{n} checked, {len(uae_scenes)} hits so far")
    print(f"\nTotal UAE urban scenes: {len(uae_scenes)}")
except Exception as e:
    print(f"Could not walk the catalog: {type(e).__name__}: {e}")
    print("Fall back on the four known IDs checked above.")

## 5 · Verdict

Fill this in from the numbers printed above and paste it to the team.

```
Landsat epoch A scenes : ____     (need >= 8, want >= 20)
Landsat epoch B scenes : ____
Sentinel-2 scenes      : ____     (need >= 5)
ESA WorldCover         : yes / no
Tanager UAE urban scene: yes / no   <- decides the hyperspectral bonus
```

**Decision rule**

- Landsat A and B both healthy → the core heat analysis is green, proceed.
- Landsat thin in epoch A → widen to 2013–2017 or raise cloud threshold to 25%.
- Tanager UAE scene found → Tier 2 hyperspectral is on; plan for it Thursday.
- No Tanager UAE scene → try EnMAP self-serve download, and if that also fails,
  document the gap honestly in README §9 Limitations. Do not invent coverage.
